7-Day Login Streak Detection
===
Difficulty: Medium

Problem Description:
===================
Given a table of user login events, find the **percentage of users** who had at least one
streak of **7 consecutive days** of activity. Round to 2 decimal places and return as a fraction
(e.g., 35.67% → 0.36).

Sample Input:
```
| user_id | login_date |
|---------|------------|
| 1       | 2024-01-01 |  ← User 1: 10 consecutive days → has streak ✅
| 1       | 2024-01-02 |
| ...     | ...        |
| 2       | 2024-01-01 |  ← User 2: gap on Jan 7 → no 7-day streak ❌
| 3       | 2024-01-01 |  ← User 3: exactly 7 days → has streak ✅
```

Sample Output:
```
percent_of_users: 0.5   (2 out of 4 users had a streak)
```

In [43]:
import pandas as pd

logins = pd.DataFrame({
    'user_id': (
        [1]*10 +  # User 1: Jan 1-10, 10 consecutive ✅
        [2]*8  +  # User 2: Jan 1-6, gap, Jan 8-9 ❌
        [3]*7  +  # User 3: exactly Jan 1-7 ✅
        [4]*5     # User 4: only 5 consecutive ❌
    ),
    'login_date': pd.to_datetime(
        [f'2024-01-{d:02d}' for d in range(1, 11)] +
        [f'2024-01-{d:02d}' for d in range(1, 7)] + ['2024-01-08', '2024-01-09'] +
        [f'2024-01-{d:02d}' for d in range(1, 8)] +
        [f'2024-01-{d:02d}' for d in range(1, 6)]
    )
})
print(logins.tail(20))

    user_id login_date
10        2 2024-01-01
11        2 2024-01-02
12        2 2024-01-03
13        2 2024-01-04
14        2 2024-01-05
15        2 2024-01-06
16        2 2024-01-08
17        2 2024-01-09
18        3 2024-01-01
19        3 2024-01-02
20        3 2024-01-03
21        3 2024-01-04
22        3 2024-01-05
23        3 2024-01-06
24        3 2024-01-07
25        4 2024-01-01
26        4 2024-01-02
27        4 2024-01-03
28        4 2024-01-04
29        4 2024-01-05


In [54]:
logins['date_rank'] = logins.groupby('user_id')['login_date'].cumcount()

logins['streak_col'] = logins['login_date'] - pd.to_timedelta(logins['date_rank'], unit='D')

streak_sizes = logins.groupby(['user_id','streak_col']).size().reset_index(name='streak_count')


ctr = (streak_sizes['streak_count'] >= 7).sum() / streak_sizes['user_id'].nunique()

# print(logins[logins['user_id'] == 2])

# logins = (logins['streak_count'] >= 7).sum()

streak_sizes


,user_id,streak_col,streak_count
0,1,2024-01-01,10
1,2,2024-01-01,6
2,2,2024-01-02,2
3,3,2024-01-01,7
4,4,2024-01-01,5


**Concepts to use (the date-minus-rank trick):**
1. **Deduplicate** first — multiple logins same day should count as one.
2. **`cumcount()`** — assigns a sequential rank (0, 1, 2...) within each user group.
3. **`date - rank`** — subtracting the rank from the date gives the **same anchor date for all days in a consecutive run**. A gap breaks the streak → anchor date changes.
4. **`groupby([user_id, streak_key]).size()`** — size of each consecutive block.
5. **Filter max streak ≥ 7** → count users → divide by total users.

In [ ]:
# Optimised Solution — Date minus Rank trick
def seven_day_streak_pct(df):
    # Step 1: deduplicate (one login per user per day)
    df = df.drop_duplicates(subset=['user_id', 'login_date'])
    df = df.sort_values(['user_id', 'login_date']).reset_index(drop=True)

    # Step 2: cumcount within each user = rank in sorted order
    df['rank'] = df.groupby('user_id').cumcount()

    # Step 3: streak_key = date - rank (same for all days in a consecutive run)
    df['streak_key'] = df['login_date'] - pd.to_timedelta(df['rank'], unit='D')

    # Step 4: size of each consecutive block
    streak_sizes = (
        df.groupby(['user_id', 'streak_key'])
          .size()
          .reset_index(name='streak_len')
    )

    # Step 5: max streak per user
    max_streak = streak_sizes.groupby('user_id')['streak_len'].max().reset_index()

    total_users   = df['user_id'].nunique()
    users_w_streak = (max_streak['streak_len'] >= 7).sum()

    return round(users_w_streak / total_users, 2)

print(f"Percent of users with 7-day streak: {seven_day_streak_pct(logins)}")